# Clase 9. Planificación y modelos: cuando el agente aprende cómo funciona el mundo

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

Bienvenida o bienvenido a la clase nueve. Hoy das un salto respecto a todo lo que has visto hasta ahora: en vez de un agente que solo reacciona a la experiencia, construyes uno que **aprende un modelo interno del entorno** y lo usa para planificar.

## Guion de la clase

1. RL sin modelo frente a RL basado en modelo
2. Dyna-Q: aprender actuando y aprender imaginando
3. MCTS, en breve, como puente hacia el capítulo 12
4. Código en vivo: Dyna-Q contra Q-learning puro en un mundo-cuadrícula con muros
5. Puntos clave y qué viene en la clase 10


## 1. RL sin modelo frente a RL basado en modelo

*Guion de clase: arranca con la analogía del mapa. Conducir a ciegas corrigiendo cada giro es RL sin modelo; abrir un mapa mental y planificar la ruta antes de arrancar es RL basado en modelo.*

Todo lo que has visto hasta la clase ocho —bandidos, Q-learning, gradientes de política, actor-crítico— pertenece a la familia de RL **sin modelo**: el agente aprende directamente una política o una función de valor a partir de la experiencia, sin representar de forma explícita cómo cambia el entorno de un paso a otro.

El RL **basado en modelo** añade una pieza: un modelo de las dinámicas del entorno, formado por una función de transición (qué estado viene después) y una función de recompensa (qué recompensa se obtiene). Ese modelo puede estar disponible de antemano o, lo más habitual, aprenderse a partir de la experiencia.

Piensa en un modelo como una aproximación de un Proceso de Decisión de Markov (MDP): no necesita ser una réplica exacta del entorno, solo lo bastante fiel como para orientar decisiones. Hay tres tipos de modelo, y cada uno falla de una manera distinta:

- **Modelo de esperanza**: predice el estado siguiente y la recompensa esperados, en promedio. Ese promedio puede no corresponder a ningún estado real (un robot que "alucina" un hueco en una pared que no existe)
- **Modelo estocástico o generativo**: genera muestras de estados y recompensas válidos, con ruido nuevo en cada consulta. Nunca devuelve un estado imposible, pero introduce más varianza
- **Modelo de dinámica completa**: aprende la distribución de probabilidad entera sobre los estados siguientes. En espacios continuos exige trabajar con integrales, algo que suele aproximarse con métodos como Markov Chain Monte Carlo (MCMC)

Un modelo puede representarse como tabla, como función lineal, como red neuronal o incluso de forma no paramétrica: un buffer de repetición de experiencias, donde "consultar el modelo" es sacar al azar una transición real que ya observaste. Ese último caso es justo el que vas a programar hoy.


## 2. Dyna-Q: aprender actuando y aprender imaginando

*Guion de clase: dibuja en la pizarra las dos fases del bucle antes de tocar código. Insiste en que Dyna-Q con cero pasos de planificación ES Q-learning puro, no una aproximación.*

La familia Dyna, propuesta por Richard Sutton, combina dos fuentes de aprendizaje que hasta ahora tratabas por separado: la experiencia real y la experiencia simulada por un modelo aprendido.

Dyna-Q, su versión tabular, funciona en dos fases dentro de cada paso:

1. **Aprendizaje directo**: el agente actúa en el entorno real, observa el estado siguiente y la recompensa, aplica una actualización de Q-learning normal y guarda esa transición en su modelo interno
2. **Planificación simulada**: después de cada paso real, el agente repite *n* veces lo siguiente: elige al azar una combinación de estado y acción que ya haya observado antes, le pregunta al modelo qué ocurriría y aplica otra actualización de Q-learning sobre esa transición imaginada

Ese número *n* se llama **pasos de planificación**. Con *n* igual a cero, Dyna-Q se reduce exactamente a Q-learning puro. Cuantos más pasos de planificación añades, más "repasa" el agente su propia experiencia antes de volver a actuar, y menos episodios reales necesita para converger, a cambio de más cómputo por paso real.

**Un matiz importante cuando el entorno cambia:**

- Si un camino que antes estaba libre se bloquea, Dyna-Q actualiza su modelo en cuanto choca contra el obstáculo nuevo y replanifica enseguida
- Si aparece un atajo nuevo que el agente nunca ha probado, Dyna-Q puede tardar en descubrirlo, porque nada le indica que existe hasta que la exploración lo lleva allí por casualidad
- **Dyna-Q+** añade un bono de exploración que crece cuanto más tiempo lleva sin visitarse una combinación de estado y acción, lo que ayuda a encontrar atajos nuevos más rápido

Un modelo aprendido nunca es perfecto. Combinar planificación basada en modelo con actualizaciones libres de modelo —cayendo en estas últimas cuando el modelo resulta poco fiable— te da lo mejor de ambos mundos: la orientación estructurada de un modelo y la robustez de aprender de la realidad.


## 3. MCTS: explorar un árbol de futuros posibles

*Guion de clase: menciona MCTS como preparación para AlphaZero en la clase 12, sin profundizar en código hoy.*

La búsqueda de Monte Carlo en árboles (Monte Carlo Tree Search, MCTS) lleva la planificación un paso más allá: en vez de simular transiciones sueltas como Dyna-Q, construye y mantiene un árbol de futuros posibles a partir del estado actual, creciendo solo por la parte que realmente explora.

Cada búsqueda se organiza en cuatro fases que se repiten tantas veces como el presupuesto de cómputo lo permita:

1. **Selección**: desde la raíz, recorre los nodos ya existentes con una política de árbol, hasta un nodo que aún no se ha expandido del todo
2. **Expansión**: añade un nuevo nodo hijo, correspondiente a una acción todavía no probada desde ahí
3. **Simulación (rollout)**: desde ese nodo nuevo, juega una trayectoria completa con una política rápida, a menudo aleatoria, hasta un estado terminal
4. **Retropropagación**: propaga el resultado hacia arriba, actualizando el valor y el contador de visitas de cada nodo del camino

El valor de un nodo es el promedio de los resultados de todas las simulaciones que pasaron por él. Upper Confidence Bound for Trees (UCT) decide qué rama explorar con la misma lógica que Upper Confidence Bound (UCB) aplicaba a los bandidos de la clase tres: suma al valor estimado un bono de exploración que crece cuando el nodo se ha visitado poco.

MCTS con UCT es el motor de búsqueda detrás de AlphaGo. Dos técnicas emparentadas, el Control Predictivo por Modelo (MPC) y MuZero, quedan fuera del alcance de esta clase.


## 4. Código en vivo: Dyna-Q contra Q-learning puro

*Guion de clase: construye el ejemplo celda a celda. Pide a la clase que prediga, antes de ejecutar la gráfica final, qué curva bajará más rápido.*

Vas a construir un mundo-cuadrícula pequeño con dos muros que obligan al agente a rodear para llegar a la meta. Vas a comparar Q-learning puro (cero pasos de planificación) con dos versiones de Dyna-Q (cinco y 50 pasos de planificación).

Solo necesitas `numpy` para el cálculo y `matplotlib` para la gráfica: ambos vienen preinstalados en Google Colab.

**Empieza importando las librerías y fijando la semilla, para que el experimento sea reproducible.**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


### El mundo-cuadrícula

Definimos una cuadrícula de seis filas por nueve columnas. El agente arranca en la esquina inferior izquierda y la meta está en la esquina superior derecha. Dos segmentos de muro, `PAREDES`, obligan a rodear en vez de ir en línea recta.

Las cuatro acciones posibles son arriba, abajo, izquierda y derecha.


In [ ]:
# --- Definimos un mundo-cuadrícula pequeño, hecho a mano ---
ALTO, ANCHO = 6, 9                 # 6 filas, 9 columnas
INICIO = (5, 0)                    # esquina inferior izquierda
META = (0, 8)                      # esquina superior derecha

# Dos segmentos de muro que obligan a rodear en vez de ir en línea recta
PAREDES = {(1, 3), (2, 3), (3, 3), (0, 6), (1, 6), (2, 6)}

# Cuatro acciones posibles: arriba, abajo, izquierda, derecha
ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
N_ACCIONES = len(ACCIONES)


### La dinámica del entorno

`paso_entorno` aplica una acción de forma determinista: si el movimiento choca contra un muro o el borde del mundo, el agente se queda donde estaba. Cada paso da una recompensa de -1, salvo al llegar a la meta, para incentivar el camino más corto.


In [ ]:
def paso_entorno(estado, accion_idx):
    """Aplica una acción de forma determinista: si el movimiento choca contra
    un muro o el borde del mundo, el agente se queda donde estaba."""
    if estado == META:
        return estado, 0.0, True
    d_fila, d_col = ACCIONES[accion_idx]
    fila, col = estado
    nueva_fila = min(max(fila + d_fila, 0), ALTO - 1)
    nueva_col = min(max(col + d_col, 0), ANCHO - 1)
    nuevo_estado = (nueva_fila, nueva_col)
    if nuevo_estado in PAREDES:
        nuevo_estado = estado
    terminado = nuevo_estado == META
    recompensa = 0.0 if terminado else -1.0   # -1 por paso incentiva el camino corto
    return nuevo_estado, recompensa, terminado


### Tabla Q y política epsilon-greedy

`indice` convierte una casilla (fila, columna) en un índice único de fila en la tabla Q. `elegir_accion` usa la misma política epsilon-greedy que ya conoces de la clase tres.


In [ ]:
def indice(estado):
    """Convierte una casilla (fila, columna) en un índice único de fila en la tabla Q."""
    return estado[0] * ANCHO + estado[1]


def elegir_accion(Q, estado, epsilon):
    """Política epsilon-greedy, la misma que usaste en la clase tres."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACCIONES)
    fila_q = Q[indice(estado)]
    mejores = np.flatnonzero(fila_q == np.max(fila_q))
    return np.random.choice(mejores)


### El agente Dyna-Q

Aquí está el corazón de la clase. Fíjate en las dos fases dentro del bucle `while`:

- **Fase 1** (aprendizaje directo): actúa, observa, actualiza Q con la experiencia real y guarda la transición en `modelo`
- **Fase 2** (planificación simulada): repite `n_planning` veces la operación de muestrear una transición ya vivida y actualizar Q con ella, sin volver a tocar el entorno real

Con `n_planning=0` no se ejecuta ninguna iteración de la fase 2: el agente queda reducido a Q-learning puro.


In [ ]:
def ejecutar_dyna_q(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Entrena un agente Dyna-Q tabular. Con n_planning=0 es Q-learning puro:
    solo queda la actualización con experiencia real, sin repetir nada."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modelo aprendido: (estado, accion) -> (estado_siguiente, recompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: aprendizaje directo, con experiencia real ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: planificación simulada, con experiencia imaginada ---
            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


### Comparamos tres configuraciones

Entrenamos con cero, cinco y 50 pasos de planificación, y promediamos diez repeticiones de cada una para suavizar el azar de la exploración.

**Ejecuta la celda y observa cuánto tarda cada configuración en calcularse: el coste extra de cómputo también es parte de la comparación.**


In [ ]:
# --- Comparamos tres configuraciones: 0, 5 y 50 pasos de planificación ---
n_episodios = 50
n_repeticiones = 10       # promediamos varias corridas para suavizar el azar
configuraciones = [0, 5, 50]
resultados = {}

for n_planning in configuraciones:
    acumulado = np.zeros(n_episodios)
    for _ in range(n_repeticiones):
        acumulado += ejecutar_dyna_q(n_planning, n_episodios)
    resultados[n_planning] = acumulado / n_repeticiones

print("Entrenamiento terminado para", configuraciones, "pasos de planificación")


### Graficamos la comparación

**Ejecuta esta celda para ver las tres curvas.**


In [ ]:
# --- Graficamos la comparación ---
plt.figure(figsize=(10, 5))
for n_planning in configuraciones:
    etiqueta = "Q-learning puro (0 pasos)" if n_planning == 0 else f"Dyna-Q ({n_planning} pasos de planificación)"
    plt.plot(resultados[n_planning], label=etiqueta)
plt.xlabel("Episodio")
plt.ylabel("Pasos hasta llegar a la meta")
plt.title("Dyna-Q vs. Q-learning puro en un mundo-cuadrícula con muros")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lectura del resultado**

*Guion de clase: para en esta gráfica y pregunta a la clase qué curva esperaban y cuál ven.*

Las tres curvas descienden, pero a ritmos muy distintos. La de cero pasos de planificación (Q-learning puro) mejora despacio, porque solo aprende de lo que va viviendo en tiempo real. Las curvas de cinco y, sobre todo, 50 pasos de planificación caen mucho más rápido: cada paso real se aprovecha muchas veces más, gracias a las actualizaciones simuladas sobre el modelo que el propio agente ha ido construyendo.


## Puntos clave

- El RL basado en modelo añade una función de transición y una función de recompensa que el agente usa para planificar, en vez de reaccionar solo a la experiencia directa
- Existen tres tipos de modelo con distintos modos de fallo: el modelo de esperanza puede predecir estados que no existen, el modelo estocástico genera siempre estados válidos a cambio de más varianza, y el modelo de dinámica completa captura toda la distribución, a mayor coste computacional
- Dyna-Q combina, en cada paso, una actualización con experiencia real y varias actualizaciones con experiencia simulada por el modelo aprendido; con cero pasos de planificación, se reduce a Q-learning puro
- Dyna-Q+ añade un bono de exploración que crece con el tiempo desde la última visita, lo que ayuda a descubrir atajos nuevos que Dyna-Q tardaría en encontrar
- MCTS explora un árbol de futuros posibles en cuatro fases (selección, expansión, simulación y retropropagación), y UCT equilibra exploración y explotación con la misma idea que UCB en los bandidos de la clase tres
- Un modelo aprendido nunca es perfecto: combinarlo con actualizaciones libres de modelo aporta robustez cuando el modelo falla

## Antes de la próxima clase

**Abre el notebook de práctica de esta clase y completa los dos ejercicios sobre Dyna-Q.** Vas a programar tú misma o tú mismo el paso de planificación simulada desde cero y vas a medir el efecto real de variar los pasos de planificación.

En la clase 10 vuelves al RL profundo sin modelo, pero con las técnicas más avanzadas del campo: Rainbow DQN y el RL distribucional.
